In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import gc
import torch
from huggingface_hub import login

# 🚀 OPTIMİZASYON: Persistent JIT Cache (Google Drive)
drive_cache = "/content/drive/MyDrive/torch_cache"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = drive_cache
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Temizlik
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)
print('✅ HuggingFace kimlik doğrulaması tamamlandı ve JIT Cache yolu ayarlandı.')


In [ ]:
%cd "/content/drive/MyDrive/obs-bologna-chatai"

print("📦 Bağımlılıklar kuruluyor...")
!pip install -r requirements.txt -q

import subprocess, sys
def check_ver(pkg):
    try: return subprocess.check_output([sys.executable, "-c", f"import {pkg}; print({pkg}.__version__)"]).decode().strip()
    except: return "Yüklenemedi"

print(f"\n✅ transformers: {check_ver('transformers')}")
print(f"✅ torch: {check_ver('torch')}")
print(f"✅ quanto: {check_ver('quanto')}")


In [ ]:
# ── FlashAttention (FA2) & JIT Optimizasyon Katmanı ──
import torch, subprocess, sys, os

cc = torch.cuda.get_device_capability()
print(f"🔍 Tespit Edilen GPU CC: {cc[0]}.{cc[1]}")

if cc[0] == 8:
    # ── A100: FlashAttention-2 Kalıcı Kurulum ──
    WHEEL_DIR = "/content/drive/MyDrive/torch_cache/fa2_wheels"
    os.makedirs(WHEEL_DIR, exist_ok=True)

    # Drive'da önceden derlenmiş wheel var mı kontrol et
    import glob
    existing_wheels = glob.glob(f"{WHEEL_DIR}/flash_attn-*.whl")

    if existing_wheels:
        # ✅ Hızlı Yol: Önceden derlenmiş wheel'den kur (~10 saniye)
        wheel_path = existing_wheels[-1]  # En son derlenen
        print(f"⚡ Kayıtlı FA2 wheel bulundu: {os.path.basename(wheel_path)}")
        !pip install {wheel_path} -q
    else:
        # 🔨 İlk Kez: Derle ve Drive'a kaydet (~15-20 dk, sadece bir kez)
        print("🔨 FA2 ilk kez derleniyor ve Google Drive'a kaydediliyor...")
        print("⏳ Bu işlem ~15-20 dakika sürer ama sadece BİR KEZ yapılır.")
        !pip install ninja packaging -q
        os.environ["MAX_JOBS"] = "4"
        !pip wheel flash-attn --no-build-isolation -w {WHEEL_DIR}
        # Derlenen wheel'i kur
        new_wheels = glob.glob(f"{WHEEL_DIR}/flash_attn-*.whl")
        if new_wheels:
            !pip install {new_wheels[-1]} -q

    # Kurulum doğrulaması
    try:
        import flash_attn
        print(f"✅ FlashAttention-2 aktif: v{flash_attn.__version__}")
    except ImportError:
        print("⚠️ FA2 kurulamadı. SDPA fallback kullanılacak.")

elif cc[0] >= 10:
    print("🚀 Blackwell algılandı. Native SDPA + JIT Cache kullanılacak.")
else:
    print(f"⚠️ GPU CC {cc[0]}.{cc[1]} — SDPA kullanılacak.")

# JIT Cache Klasörünü Oluştur
cache_path = os.environ.get("TORCHINDUCTOR_CACHE_DIR")
if cache_path:
    os.makedirs(cache_path, exist_ok=True)
    print(f"✅ JIT Cache dizini hazır: {cache_path}")

In [ ]:
# FlashAttention-3 & Environment Verification
import torch
import transformers
from transformers.utils import is_flash_attn_3_available
import sys
sys.path.append('.')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Capability: {torch.cuda.get_device_capability(0)}")

try:
    import flash_attn_interface
    print("✅ flash-attn-3 (flash_attn_interface) found.")
except ImportError:
    print("❌ flash-attn-3 NOT found.")

print(f"Transformers FA3 available (pre-patch): {is_flash_attn_3_available()}")

try:
    from src.model import RAGModel
    print(f"Transformers FA3 available (post-patch): {is_flash_attn_3_available()}")
except Exception as e:
    print(f"Error importing RAGModel: {e}")


In [ ]:
import torch
import transformers

print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    bf16 = torch.cuda.is_bf16_supported()
    sdpa = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
    print(f'✅ GPU   : {gpu}')
    print(f'✅ VRAM  : {vram:.1f} GB')
    print(f'✅ BF16  : {"Destekleniyor" if bf16 else "DESTEKLENMIYOR"}')
    print(f'✅ SDPA  : {"Aktif" if sdpa else "Yok"}')

    try:
        from transformers import Gemma4ForConditionalGeneration
        print('✅ Gemma4 Model : TAMAM')
    except ImportError as e:
        print(f'❌ Gemma4 Model : {e}')

    try:
        from transformers import BertModel
        print('✅ BertModel    : TAMAM')
    except ImportError as e:
        print(f'❌ BertModel    : {e}')
else:
    print('⚠️  GPU bulunamadı!')


In [ ]:
import torch
import sys
import os

# 1. Donanım Doğrulama
cc = torch.cuda.get_device_capability()
gpu_name = torch.cuda.get_device_name(0)
print(f"🔍 Cihaz: {gpu_name} (SM {cc[0]}.{cc[1]}")

# 2. torch.compile Sanity Check
print("\n⚙️ torch.compile doğrulama testi başlatılıyor...")
try:
    def simple_fn(x): return x * 2 + 1
    compiled_fn = torch.compile(simple_fn)
    test_tensor = torch.randn(2, 2).cuda()
    out = compiled_fn(test_tensor)
    print("✅ torch.compile kernel derlemesi başarılı.")
except Exception as e:
    print(f"❌ torch.compile hatası: {e}")

# 3. Dosya Senkronizasyon Kontrolü
print("\n📁 v6.0 Academic Edition Dosya Kontrolü...")
if os.path.exists("main.py") and os.path.exists("src/metrics.py"):
    print("✅ Proje dosyaları senkronize durumda.")
else:
    print("❌ Eksik dosyalar var, Google Drive senkronizasyonunu bekleyin.")


In [ ]:
!python main.py


In [ ]:
# 🚀 FAZ 4: YENİ EMBEDDING VE MASTER-CHUNKING TABANLI BENCHMARK ÇALIŞTIRMA
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("🧹 VRAM temizlendi. Benchmark süreci başlatılıyor...")

!python test_inference.py --benchmark


In [ ]:
import json, glob
import pandas as pd

metric_files = sorted(glob.glob('data/metrics/*.json'))
if not metric_files:
    print('Henüz metrik kaydı yok. Önce benchmark çalıştırın.')
else:
    records = [json.load(open(f)) for f in metric_files]
    df = pd.DataFrame(records)
    display_cols = [
        'timestamp', 'model_name', 'gpu',
        'ttft_seconds', 'total_time_seconds', 'tokens_per_second',
        'faithfulness_score', 'rouge_l_f1',
        'context_token_count', 'retrieved_doc_count',
    ]
    display(df[[c for c in display_cols if c in df.columns]].to_string(index=False))


In [ ]:
# ── RAG Metrik Analizi (Filtrelenmiş) ──
import json, glob
import pandas as pd

metric_files = sorted(glob.glob('data/metrics/*.json'))
if not metric_files:
    print('Metrik bulunamadı. Lütfen benchmark hücresini çalıştırın.')
else:
    records = []
    for f in metric_files:
        try:
            with open(f, encoding='utf-8') as file:
                records.append(json.load(file))
        except Exception:
            pass
    df = pd.DataFrame(records)

    # Sadece başarılı (süre hesabı yapılmış) kayıtları filtrele
    df_clean = df[df['total_time_seconds'].notna()].copy()

    # Son 10 testi göster
    cols = ['timestamp', 'benchmark_version', 'query', 'faithfulness_score', 'total_time_seconds', 'retrieved_doc_count']
    valid_cols = [c for c in cols if c in df_clean.columns]
    display(df_clean[valid_cols].tail(10))

    # All-Time
    all_time_mean = df_clean['faithfulness_score'].mean()
    print(f"\n📈 All-Time Average Faithfulness: {all_time_mean:.4f}")

    # Current Version (v4.2)
    target_version = "v4.2"
    if 'benchmark_version' in df_clean.columns:
        df_ver = df_clean[df_clean['benchmark_version'] == target_version]
        if not df_ver.empty:
            ver_mean = df_ver['faithfulness_score'].mean()
            print(f"🎯 Version {target_version} Average Faithfulness: {ver_mean:.4f} (n={len(df_ver)})")
        else:
            print(f"🎯 Version {target_version} Average Faithfulness: Henüz kayıt yok.")
    else:
        print(f"🎯 Version {target_version} Average Faithfulness: benchmark_version alanı bulunamadı (Eski kayıtlar).")

    # Last Run (Last 5 records)
    if len(df_clean) >= 5:
        last_run_mean = df_clean['faithfulness_score'].tail(5).mean()
        print(f"⚡ Last Run (5 Queries) Average Faithfulness: {last_run_mean:.4f}")
    elif not df_clean.empty:
        last_run_mean = df_clean['faithfulness_score'].mean()
        print(f"⚡ Last Run (All {len(df_clean)} Queries) Average Faithfulness: {last_run_mean:.4f}")
